In [1]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta

fake = Faker('en_IN')
np.random.seed(42)
random.seed(42)

print("Setup complete!")

Setup complete!


In [2]:
def generate_customers(n=1000):
    cities = ['Mumbai', 'Delhi', 'Bangalore', 'Hyderabad', 
              'Chennai', 'Pune', 'Nagpur', 'Kolkata']
    
    income_levels = ['Low', 'Medium', 'High', 'Premium']
    balance_ranges = {
        'Low':     (5000,   50000),
        'Medium':  (50000,  200000),
        'High':    (200000, 800000),
        'Premium': (800000, 5000000)
    }
    
    customers = []
    for i in range(n):
        income = random.choice(income_levels)
        low, high = balance_ranges[income]
        
        customers.append({
            'customer_id':  f'C{i+1:04d}',
            'name':         fake.name(),
            'age':          random.randint(22, 65),
            'gender':       random.choice(['Male', 'Female']),
            'city':         random.choice(cities),
            'income_level': income,
            'account_type': random.choice(['Savings', 'Current', 'Salary']),
            'balance':      round(random.uniform(low, high), 2)
        })
    
    return pd.DataFrame(customers)

customers_df = generate_customers(1000)
customers_df.head()

,customer_id,name,age,gender,city,income_level,account_type,balance
0,C0001,Chasmum Bath,23,Female,Hyderabad,Low,Savings,11279.21
1,C0002,Karan Palla,65,Male,Nagpur,Low,Savings,6340.87
2,C0003,Vedant Minhas,36,Male,Hyderabad,Medium,Salary,147482.67
3,C0004,Harita Chad,36,Female,Chennai,Premium,Savings,3986990.94
4,C0005,Daksha Sawhney,49,Female,Chennai,Medium,Savings,82297.06


In [3]:
def generate_campaigns():
    campaigns = [
        {'campaign_id': 'CAM001', 'campaign_name': 'Savings Booster',
         'campaign_type': 'Product Promotion', 'channel': 'Email',
         'target_segment': 'Low', 'start_date': '2024-01-15'},
        
        {'campaign_id': 'CAM002', 'campaign_name': 'Investment Plus',
         'campaign_type': 'Cross Sell', 'channel': 'SMS',
         'target_segment': 'High', 'start_date': '2024-02-10'},
        
        {'campaign_id': 'CAM003', 'campaign_name': 'Cashback Offer',
         'campaign_type': 'Retention', 'channel': 'App Notification',
         'target_segment': 'Medium', 'start_date': '2024-03-05'},
        
        {'campaign_id': 'CAM004', 'campaign_name': 'Premium Wealth',
         'campaign_type': 'Upsell', 'channel': 'Email',
         'target_segment': 'Premium', 'start_date': '2024-04-01'},
        
        {'campaign_id': 'CAM005', 'campaign_name': 'Salary Account Drive',
         'campaign_type': 'Acquisition', 'channel': 'SMS',
         'target_segment': 'Medium', 'start_date': '2024-05-20'},
    ]
    return pd.DataFrame(campaigns)

campaigns_df = generate_campaigns()
campaigns_df.head()

,campaign_id,campaign_name,campaign_type,channel,target_segment,start_date
0,CAM001,Savings Booster,Product Promotion,Email,Low,2024-01-15
1,CAM002,Investment Plus,Cross Sell,SMS,High,2024-02-10
2,CAM003,Cashback Offer,Retention,App Notification,Medium,2024-03-05
3,CAM004,Premium Wealth,Upsell,Email,Premium,2024-04-01
4,CAM005,Salary Account Drive,Acquisition,SMS,Medium,2024-05-20


In [8]:
def generate_responses(customers_df, campaigns_df, n=5000):
    responses = []
    
    conversion_rates = {
        True:  0.35,  # matched → high conversion
        False: 0.08   # mismatched → low conversion
    }
    
    for i in range(n):
        campaign = campaigns_df.sample(1).iloc[0]
        
        # 70% chance: pick customer from the TARGET segment (match)
        # 30% chance: pick any random customer (mismatch)
        if random.random() < 0.70:
            target_customers = customers_df[
                customers_df['income_level'] == campaign['target_segment']
            ]
            customer = target_customers.sample(1).iloc[0]
        else:
            customer = customers_df.sample(1).iloc[0]
        
        is_match = customer['income_level'] == campaign['target_segment']
        converted = random.random() < conversion_rates[is_match]
        
        response_date = datetime.strptime(campaign['start_date'], '%Y-%m-%d') \
                        + timedelta(days=random.randint(1, 30))
        
        responses.append({
            'response_id':   f'R{i+1:05d}',
            'customer_id':   customer['customer_id'],
            'campaign_id':   campaign['campaign_id'],
            'response_type': random.choice(['Opened', 'Clicked', 'Ignored', 'Converted', 'Spam']),
            'converted':     converted,
            'response_date': response_date.strftime('%Y-%m-%d')
        })
    
    return pd.DataFrame(responses)

# Regenerate with the fix
responses_df = generate_responses(customers_df, campaigns_df)

In [9]:
import os

customers_df.to_csv('../data/raw/customers.csv', index=False)
campaigns_df.to_csv('../data/raw/campaigns.csv', index=False)
responses_df.to_csv('../data/raw/responses.csv', index=False)

print(f"Customers: {len(customers_df)} rows")
print(f"Campaigns: {len(campaigns_df)} rows")
print(f"Responses: {len(responses_df)} rows")
print("All files saved to data/raw/")


Customers: 1000 rows
Campaigns: 5 rows
Responses: 5000 rows
All files saved to data/raw/


In [11]:
# How many customers converted per campaign?
import pandas as pd

merged = responses_df.merge(customers_df, on='customer_id')
merged = merged.merge(campaigns_df, on='campaign_id')

result = merged.groupby('campaign_name')['converted'].agg(['sum', 'count', 'mean'])
result.columns = ['conversions', 'total_responses', 'conversion_rate']
result['conversion_rate'] = result['conversion_rate'].round(2)
print(result)


                      conversions  total_responses  conversion_rate
campaign_name                                                      
Cashback Offer                289              988             0.29
Investment Plus               306              969             0.32
Premium Wealth                307             1021             0.30
Salary Account Drive          264              988             0.27
Savings Booster               265             1034             0.26
